# Bayesian Filtering

## Thinking About Belief
Baye's Law is a mechanism for updating our prior beliefs based on the presentation of new evidence. Bayesian filtering is a family of algorithms for performing these updates of belief over time as we receive new information.

In Bayesian language, we describe belief as a probability distribution rather than trying to guess some single, discrete value. Instead of thinking things like

> I think we are going X miles an hour

We describe our belief with phrases like

> I think we are going between X and Y miles per hour

This second way of framing belief aligns with our intuition about probability distributions. Probability distributions tell us about a range of potential outcomes rather than specifying a single one. 

## Notation

Since we are talking about probability, we will distinguish random variables from their realized, concrete values. 

$$
\begin{align}
X_t &:= \text{A random variable representing our unkown position at time } t \\
x_t &:= \text{A realized value of our random variable at time } t
\end{align}
$$

## Bayes Law

I'll put this here now, but we aren't going to talk about how it relates to bayesian filtering. That will be made apparent as we step through the algorithm. 

$$
P(H | E) = \frac{P( E | H ) P(H)}{P(E)}
$$

## The Algorithm

### Step 1: Establishing Prior Belief
Let's say we are concerned with ascertaining our position over time. At time 0 we might have some belief about where we are, expressed as a probability distribution:

$$
P(X_0)
$$

This initial belief could be an educated guess, or it might come from some measurement. 

### Step 2: Receiving New Evidence
At this point, we get a measurement from a sensor, perhaps GPS, telling us that our location is $z_0$. Note that the notation is lower case, and therefore is not a random variable but an actual measured value. 

### Step 3: Likelihood
Likelihood answers the question 

> "If I knew the true position was $\hat{x}_t$, how likely is the measurement $z_t$ I recieved?" 

In order to define likelihood mathematically, we consider that our measurement was generated by some imperfect process. For example, we might model the process of generating GPS measurement as our true (but unknown!) position $\hat{x}_t$, plus some random error $V$ that follows a normal distribution with zero mean and some standard deviation $\sigma$:

$$
\begin{align}
Z_t = \hat{x}_t + V, \\
V \sim \mathcal{N}(0, \sigma)
\end{align}
$$

What does that tell us about the distribution of $Z_t$? Since the mean value of the error of the GPS is zero, that means 

$$
Z_t \sim \mathcal{N}(\hat{x}_t, \sigma)
$$

Using this information we can construct the likelihood function. In our Gaussian example, we know the probability distribution is defined as

$$
p(z; \mu_z, \sigma) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp \bigg(-\frac{(z - \mu_z)^2}{2 \sigma^2} \bigg)
$$

Note that normally our average $\mu_z$ and standard deviation $\sigma$ is some fixed value and the only variable in question is $z$. That function gives us the probability density associated with different values of $z$. 

But in our case, we determined that the mean of the distribution of $Z_t$ is actually our unknown true state. We also obtained a measurement $Z = z_t$. If we further fix the standard deviation, we now only have a function of $x_t$, and that function is what we call likelihood.

Again, this function describes 
> "how likely is it that the GPS reported $z_t$ when the true state is $x_t$.

Another way to think about it is 

> "How much did the GPS measurement agree with my hypothesis about our location" .




In [22]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import fixed

def normal_pdf(x, mean=0.0, std=1.0):
    variance = std ** 2
    denominator = np.sqrt(2 * np.pi * variance)
    
    numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
    return numerator / denominator

@widgets.interact(mean1=widgets.FloatSlider(value=None, min=-20, max=20), 
                  std1=widgets.FloatSlider(value=1, min=0.0001, max=5), 
                  mean2=widgets.FloatSlider(value=None, min=-20, max=20), 
                  std2=widgets.FloatSlider(value=1, min=0.0001, max=5))
def update(mean1, mean2, std1, std2):
    start = -35
    stop = 35
    x = np.arange(start, stop, 0.1)
    prior = normal_pdf(x, mean1, std1)
    likelihood = normal_pdf(x, mean2, std2)
    update = np.multiply(prior, likelihood)

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.lineplot(x=x, y=prior, ax=ax, label="prior")
    sns.lineplot(x=x, y=likelihood, ax=ax, label="likelihood")
    sns.lineplot(x=x, y=update, ax=ax, label="update")
    # plt.tight_layout()
    plt.show()



interactive(children=(FloatSlider(value=0.0, description='mean1', max=20.0, min=-20.0), FloatSlider(value=0.0,…